In [1]:
import pandas as pd
from surprise import SVD, Dataset, Reader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [ ]:
ratings = pd.read_csv("ml-32m/ratings.csv")
movies = pd.read_csv("ml-32m/movies.csv")
tags = pd.read_csv("ml-32m/tags.csv")

In [ ]:
# Merging tags into movies for content features
tags['tag'] = tags['tag'].fillna('').astype(str)
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movie_data = movies.merge(tags_grouped, on='movieId', how='left')
movie_data['tag'] = movie_data['tag'].fillna('')
movie_data['combined'] = movie_data['genres'] + ' ' + movie_data['tag']

# TF-IDF on combined genres + tags
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movie_data['combined'])

In [ ]:
#KNN model
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(tfidf_matrix)

#Surprise SVD model
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()
svd_model = SVD()
svd_model.fit(trainset)

In [7]:
def hybrid_recommend(user_id, target_movie_title, N=5, alpha=0.7):
    # Finding the movieId for the target movie
    target_match = movie_data[movie_data["title"].str.contains(target_movie_title, case=False, na=False)]
    if target_match.empty:
        return f"Movie '{target_movie_title}' not found."
    target_movie_id = target_match['movieId'].iloc[0]
    target_index = movie_data[movie_data["movieId"] == target_movie_id].index[0]

    # Getting KNN similarity scores
    distances, indices = knn_model.kneighbors(tfidf_matrix[target_index], n_neighbors=100)
    knn_scores = [(movie_data.iloc[idx]['movieId'], 1 - dist) for dist, idx in zip(distances[0], indices[0])]
    knn_score_dict = dict(knn_scores)

    # Getting SVD predictions for all movies for this user
    all_movie_ids = movie_data['movieId'].tolist()
    svd_preds = [svd_model.predict(str(user_id), str(mid)) for mid in all_movie_ids]

    # Combining the scores
    hybrid_results = []
    for pred in svd_preds:
        mid = int(pred.iid)
        if mid == target_movie_id:
            continue  
        svd_score = pred.est
        knn_sim = knn_score_dict.get(mid, 0)
        hybrid_score = alpha * svd_score + (1 - alpha) * knn_sim
        hybrid_results.append((mid, hybrid_score))

    # Sorting to get top N
    top = sorted(hybrid_results, key=lambda x: x[1], reverse=True)[:N]
    top_ids = [mid for mid, _ in top]
    recommendations = movie_data[movie_data['movieId'].isin(top_ids)][['movieId', 'title']]

    print(f"\nTop {N} Hybrid Recommendations for User {user_id} based on '{target_movie_title}':")
    return recommendations.reset_index(drop=True)


In [8]:
hybrid_recommend(user_id=1, target_movie_title="Toy Story", N=5, alpha=0.7)



Top 5 Hybrid Recommendations for User 1 based on 'Toy Story':


,movieId,title
0,2355,"Bug's Life, A (1998)"
1,3114,Toy Story 2 (1999)
2,4886,"Monsters, Inc. (2001)"
3,78499,Toy Story 3 (2010)
4,157296,Finding Dory (2016)
